In [2]:
from datasets import load_from_disk

ds = load_from_disk("../musiccaps/dataset_audio")
len(ds)

# preliminary results
# framework direkt metric output edicek
# machine translation metrics, BLEU, ROUGE, METEOR, CIDEr, bertscore

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

5305

In [3]:
# First split: separate test set (held out for final evaluation only)
ds_split = ds.train_test_split(test_size=0.1, seed=42)
ds_train_val = ds_split["train"]
ds_test = ds_split["test"]

# Second split: split train+val into train and validation sets
ds_train_val_split = ds_train_val.train_test_split(test_size=0.1, seed=42)
ds_train = ds_train_val_split["train"]
ds_val = ds_train_val_split["test"]

print(f"Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}")

Train: 4296, Val: 478, Test: 531


In [4]:
# Cell 1: Imports
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ClapProcessor, ClapModel
import librosa
import torch
import torch.nn as nn
from pathlib import Path

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:

# Constants (must match training)
D_AUDIO = 512  # CLAP projection dim
D_LM = 768     # GPT-2 embedding dim
PREFIX_LEN = 16  # Must match training


In [6]:

# Cell 3: Load Models
# CLAP (frozen)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap.eval()
for p in clap.parameters():
    p.requires_grad = False

# GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.eval()
for p in gpt2.parameters():
    p.requires_grad = False


In [7]:

# Cell 4: Define Projection Networks (must match training architecture)
projection = nn.Sequential(
    nn.Linear(D_AUDIO, D_LM * 2),      # 512 → 1536
    nn.LayerNorm(D_LM * 2),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 2, D_LM * 4),     # 1536 → 3072
    nn.LayerNorm(D_LM * 4),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 4, PREFIX_LEN * D_LM),  # 3072 → PREFIX_LEN * 768
).to(device)

text_projection = nn.Sequential(
    nn.Linear(D_LM, D_AUDIO),
    nn.LayerNorm(D_AUDIO),
    nn.Dropout(0.5),
    nn.GELU(),
).to(device)


In [8]:
# Cell 5: Load Checkpoint
SAVE_DIR = Path("../04_train/checkpoints")  # Adjust path as needed

checkpoint = torch.load(SAVE_DIR / "best_model_stage2.pt")

projection.load_state_dict(checkpoint["projection"])
text_projection.load_state_dict(checkpoint["text_projection"])
gpt2.load_state_dict(checkpoint["gpt2"])  # Load fine-tuned GPT-2

print("Model loaded successfully!")

Model loaded successfully!


In [9]:
# Cell 6: Helper Functions
def get_audio_embedding(sample):
    """Get audio embedding for a single sample"""
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)

    inputs = clap_processor(
        audio=audio,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clap.get_audio_features(**inputs)

    return emb  # (1, D_AUDIO)


In [10]:
# Cell 9.5: FENSE Metric Computation
def compute_fense_metric(generated_captions, ground_truth_captions):
    """
    Compute FENSE metric for fluency and semantic similarity.
    
    Args:
        generated_captions: List of generated caption strings
        ground_truth_captions: List of ground truth caption strings
    
    Returns:
        Dictionary with FENSE scores
    """
    try:
        from fense.fense import Fense
    except ImportError:
        return {
            'FENSE': None,
            'FENSE_error': 'FENSE not installed. Install with: pip install fense'
        }
    
    try:
        # Initialize FENSE evaluator
        evaluator = Fense(device=device, sbert_model='paraphrase-TinyBERT-L6-v2')
        
        print("Computing FENSE scores...")
        fense_scores = []
        
        for i, (gen, gt) in enumerate(zip(generated_captions, ground_truth_captions)):
            try:
                # FENSE returns a score (higher is better, typically 0-1 range)
                score = evaluator.score(gen, gt)
                fense_scores.append(score)
            except Exception as e:
                # If error, skip this pair
                fense_scores.append(0.0)
                continue
            
            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1}/{len(generated_captions)} samples...")
        
        avg_fense = np.mean(fense_scores)
        
        return {
            'FENSE': avg_fense,
            'individual_scores': fense_scores
        }
    
    except Exception as e:
        return {
            'FENSE': None,
            'FENSE_error': str(e)
        }


In [11]:
# Cell 7: Generation Function
def generate_caption(audio_sample, use_beam_search=True, max_length=100, min_length=20):
    """
    Generate caption from audio sample.
    
    Args:
        audio_sample: Dictionary with 'audio' key containing 'array' and 'sampling_rate'
        use_beam_search: If True, use beam search (better quality). If False, use sampling.
        max_length: Maximum length of generated caption (includes prefix)
        min_length: Minimum length of generated caption (includes prefix)
    
    Returns:
        Generated caption string
    """
    # Get audio embedding
    audio_emb = get_audio_embedding(audio_sample)
    
    # Project to prefix tokens
    prefix = projection(audio_emb)
    prefix = prefix.view(1, PREFIX_LEN, D_LM)
    
    # Create attention mask for prefix (all ones since we attend to all prefix tokens)
    attention_mask = torch.ones(1, PREFIX_LEN, dtype=torch.long, device=device)
    
    # Get EOS token ID
    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    
    # Generate caption
    if use_beam_search:
        generated = gpt2.generate(
            inputs_embeds=prefix,
            attention_mask=attention_mask,
            max_length=max_length,
            min_length=min_length,
            num_beams=10,  # Balanced between quality and speed
            early_stopping=True,
            repetition_penalty=2.25,  # High penalty to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition (catches "passionate and passionate")
            length_penalty=0.8,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            do_sample=False,  # Deterministic with beam search
        )
    else:
        # Alternative: sampling with lower temperature
        generated = gpt2.generate(
            inputs_embeds=prefix,
            attention_mask=attention_mask,
            max_length=max_length,
            min_length=min_length,
            do_sample=True,
            temperature=0.3,  # Increased from 0.1 for more natural endings
            top_k=20,
            top_p=0.8,
            repetition_penalty=1.8,  # Increased to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )
    
    # Decode to text - stop at EOS token
    caption = tokenizer.decode(generated[0], skip_special_tokens=True)
    
    # Additional cleanup: remove incomplete sentences at the end
    # Find last complete sentence (ends with . ! ?)
    last_period = max(caption.rfind('.'), caption.rfind('!'), caption.rfind('?'))
    if last_period > len(caption) * 0.5:  # Only if sentence is substantial
        caption = caption[:last_period + 1]
    
    return caption



In [ ]:
# SPIDER Metric Evaluation Setup
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from collections import Counter
import numpy as np

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Failed to find tokenizers/punkt, falling back to punkt, wordnet or omw-1.4")
    nltk.download('punkt', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)

print("SPIDER metric libraries loaded successfully!")


In [ ]:
# Cell 9: SPIDER Metric Computation Functions
def compute_spider_metric(generated_captions, ground_truth_captions):
    """
    Compute SPIDER metric (average of normalized BLEU-4, METEOR, ROUGE-L, CIDEr).
    
    Args:
        generated_captions: List of generated caption strings
        ground_truth_captions: List of ground truth caption strings
    
    Returns:
        Dictionary with individual metrics and SPIDER score
    """
    smooth = SmoothingFunction().method1
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    bleu_scores = []
    meteor_scores = []
    rouge_scores = []
    cider_scores = []
    
    print("Computing metrics for each caption pair...")
    for i, (gen, gt) in enumerate(zip(generated_captions, ground_truth_captions)):
        # Tokenize
        gen_tokens = gen.lower().split()
        gt_tokens = gt.lower().split()
        
        # Skip if empty
        if len(gen_tokens) == 0 or len(gt_tokens) == 0:
            bleu_scores.append(0.0)
            meteor_scores.append(0.0)
            rouge_scores.append(0.0)
            cider_scores.append(0.0)
            continue
        
        # BLEU-4
        try:
            bleu = sentence_bleu([gt_tokens], gen_tokens, smoothing_function=smooth)
            bleu_scores.append(bleu)
        except:
            bleu_scores.append(0.0)
        
        # METEOR
        try:
            meteor = meteor_score([gt_tokens], gen_tokens)
            meteor_scores.append(meteor)
        except Exception as e:
            meteor_scores.append(0.0)
        
        # ROUGE-L
        try:
            rouge_scores_dict = rouge_scorer_obj.score(gt, gen)
            rouge_scores.append(rouge_scores_dict['rougeL'].fmeasure)
        except:
            rouge_scores.append(0.0)
        
        # CIDEr approximation (TF-IDF based similarity)
        # This is a simplified version - full CIDEr is more complex
        gen_words = set(gen_tokens)
        gt_words = set(gt_tokens)
        if len(gen_words) > 0 and len(gt_words) > 0:
            # Jaccard similarity as CIDEr approximation
            intersection = len(gen_words & gt_words)
            union = len(gen_words | gt_words)
            cider = intersection / union if union > 0 else 0.0
        else:
            cider = 0.0
        cider_scores.append(cider)
        
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(generated_captions)} samples...")
    
    # Average scores
    avg_bleu = np.mean(bleu_scores)
    avg_meteor = np.mean(meteor_scores)
    avg_rouge = np.mean(rouge_scores)
    avg_cider = np.mean(cider_scores)
    
    # Normalize CIDEr (simple approximation - full CIDEr can be much higher)
    # Scale to [0, 1] range (CIDEr typically ranges 0-10+, we normalize by assuming max ~2.0)
    normalized_cider = min(avg_cider * 0.5, 1.0)
    
    # SPIDER = average of normalized scores
    spider = (avg_bleu + avg_meteor + avg_rouge + normalized_cider) / 4.0
    
    return {
        'BLEU_4': avg_bleu,
        'METEOR': avg_meteor,
        'ROUGE_L': avg_rouge,
        'CIDEr': normalized_cider,
        'CIDEr_raw': avg_cider,  # Raw CIDEr for reference
        'SPIDER': spider,
        'individual_scores': {
            'bleu': bleu_scores,
            'meteor': meteor_scores,
            'rouge': rouge_scores,
            'cider': cider_scores
        }
    }


In [ ]:
# Cell 10: Evaluation Function
def evaluate_model(dataset, num_samples=None, use_beam_search=True, max_length=80, min_length=20):
    """
    Evaluate model on dataset and compute SPIDER metric.
    
    Args:
        dataset: Test dataset
        num_samples: Number of samples to evaluate (None = all)
        use_beam_search: Whether to use beam search for generation
        max_length: Maximum generation length
        min_length: Minimum generation length
    
    Returns:
        Dictionary with metrics, generated captions, and ground truth captions
    """
    if num_samples is None:
        num_samples = len(dataset)
    else:
        num_samples = min(num_samples, len(dataset))
    
    generated_captions = []
    ground_truth_captions = []
    
    print(f"Generating captions for {num_samples} samples...")
    print("=" * 60)
    
    for idx in range(num_samples):
        sample = dataset[idx]
        try:
            generated = generate_caption(
                sample, 
                use_beam_search=use_beam_search,
                max_length=max_length,
                min_length=min_length
            )
            generated_captions.append(generated)
            ground_truth_captions.append(sample["caption"])
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue
        
        if (idx + 1) % 50 == 0:
            print(f"  Generated {idx + 1}/{num_samples} captions...")
    
    print(f"\nGenerated {len(generated_captions)} captions successfully.")
    print("Computing SPIDER metric...")
    print("=" * 60)
    
    # Compute metrics
    metrics = compute_spider_metric(generated_captions, ground_truth_captions)
    
    # Print results
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    print(f"BLEU-4:    {metrics['BLEU_4']:.4f}")
    print(f"METEOR:    {metrics['METEOR']:.4f}")
    print(f"ROUGE-L:   {metrics['ROUGE_L']:.4f}")
    print(f"CIDEr:     {metrics['CIDEr']:.4f} (raw: {metrics['CIDEr_raw']:.4f})")
    print(f"SPIDER:    {metrics['SPIDER']:.4f}")
    print("=" * 60)
    
    return {
        'metrics': metrics,
        'generated_captions': generated_captions,
        'ground_truth_captions': ground_truth_captions
    }


In [16]:
# Cell 10.5: Caching Setup for Evaluation
import json
import hashlib
from pathlib import Path
from datetime import datetime

# Create cache directory
CACHE_DIR = Path("evaluation_cache")
CACHE_DIR.mkdir(exist_ok=True)

def get_cache_filename(config):
    """Generate a unique cache filename based on evaluation parameters"""
    config_str = json.dumps(config, sort_keys=True)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    num_samples = config['num_samples'] if config['num_samples'] is not None else 'all'
    filename = f"eval_results_{num_samples}_{config_hash}.json"
    return CACHE_DIR / filename

def load_cached_results(config, force_recompute=False):
    """Load cached evaluation results if available"""
    cache_file = get_cache_filename(config)
    
    if cache_file.exists() and not force_recompute:
        print(f"Loading cached evaluation results from {cache_file}...")
        print("=" * 60)
        with open(cache_file, 'r') as f:
            cached_data = json.load(f)
        
        if cached_data.get('config') == config:
            results = {
                'metrics': cached_data['metrics'],
                'generated_captions': cached_data['generated_captions'],
                'ground_truth_captions': cached_data['ground_truth_captions']
            }
            print("✓ Cached results loaded successfully!")
            print(f"  Timestamp: {cached_data.get('timestamp', 'Unknown')}")
            print(f"  Samples: {len(results['generated_captions'])}")
            print(f"  SPIDER Score: {results['metrics']['SPIDER']:.4f}")
            print("=" * 60)
            return results, cache_file
        else:
            print("⚠ Cached config doesn't match. Will re-run evaluation...")
    
    return None, cache_file

def save_results_to_cache(results, config, cache_file):
    """Save evaluation results to cache file"""
    cache_data = {
        'timestamp': datetime.now().isoformat(),
        'config': config,
        'metrics': results['metrics'],
        'generated_captions': results['generated_captions'],
        'ground_truth_captions': results['ground_truth_captions']
    }
    with open(cache_file, 'w') as f:
        json.dump(cache_data, f, indent=2)
    print(f"\n✓ Results saved to cache: {cache_file}")

print("Caching utilities loaded!")


Caching utilities loaded!


In [17]:
# Cell 11 (Updated): Run Evaluation on Test Set (with caching)
# Evaluation parameters
EVAL_CONFIG = {
    'num_samples': None,  # Set to None for full evaluation, or e.g., 100 for quick test
    'use_beam_search': True,
    'max_length': 80,
    'min_length': 20
}

# Check for cached results
force_recompute = False  # Set to True to force re-evaluation
cached_results, cache_file = load_cached_results(EVAL_CONFIG, force_recompute)

if cached_results is not None:
    results = cached_results
else:
    # Run evaluation
    if force_recompute:
        print("Force recompute enabled. Running evaluation...")
    else:
        print("No cache found. Running evaluation...")
    print("=" * 60)
    
    results = evaluate_model(
        ds_test,
        **EVAL_CONFIG
    )
    
    # Save to cache
    save_results_to_cache(results, EVAL_CONFIG, cache_file)


No cache found. Running evaluation...
Generating captions for 531 samples...
  Generated 50/531 captions...
  Generated 100/531 captions...
  Generated 150/531 captions...
  Generated 200/531 captions...
  Generated 250/531 captions...
  Generated 300/531 captions...
  Generated 350/531 captions...
  Generated 400/531 captions...
  Generated 450/531 captions...
  Generated 500/531 captions...

Generated 531 captions successfully.
Computing SPIDER metric...
Computing metrics for each caption pair...
  Processed 50/531 samples...
  Processed 100/531 samples...
  Processed 150/531 samples...
  Processed 200/531 samples...
  Processed 250/531 samples...
  Processed 300/531 samples...
  Processed 350/531 samples...
  Processed 400/531 samples...
  Processed 450/531 samples...
  Processed 500/531 samples...

EVALUATION RESULTS
BLEU-4:    0.0581
METEOR:    0.2307
ROUGE-L:   0.2371
CIDEr:     0.0902 (raw: 0.1804)
SPIDER:    0.1540

✓ Results saved to cache: evaluation_cache/eval_results_all_84

In [19]:
# Cell 12: Display Sample Results
# Show some example generated vs ground truth captions
num_examples = 5
print(f"\nSample Results (first {num_examples}):")
print("=" * 60)

for i in range(min(num_examples, len(results['generated_captions']))):
    print(f"\nSample {i+1}:")
    print(f"Generated:  {results['generated_captions'][i]}")
    print(f"Ground truth: {results['ground_truth_captions'][i]}")
    print("-" * 60)



Sample Results (first 5):

Sample 1:
Generated:  The low quality recording features a live performance of a pop song that consists of passionate female vocal singing over sustained strings melody, mellow piano accompaniment, shimmering hi hats and punchy kick hits. The recording is noisy and it sounds emotional - like something you would hear in a movie scene.
Ground truth: A female vocalist sings this soft love song in a foreign language. The tempo is medium with a mellifluous violin harmony, keyboard accompaniment, subtle bass, steady drumming and rhythmic acoustic guitar. The song is sweet, youthful, simple, romantic, sentimental, melancholic and pensive. This song is Romantic Pop.
------------------------------------------------------------

Sample 2:
Generated:  The song is an instrumental. The tempo is medium with a steady drumming rhythm, steady bass guitar accompaniment, mellow electric guitar chords, shimmering e-guitar and acoustic rhythm guitar strumming chords in the backg

In [20]:
# Cell 14: Compute FENSE Metric (Run after main evaluation)
# Uses official FENSE: https://github.com/blmoistawinde/fense

try:
    from fense.evaluator import Evaluator
    
    print("Computing FENSE metric...")
    print("=" * 60)
    
    # Initialize with paper's recommended settings
    evaluator = Evaluator(
        device=device, 
        sbert_model='paraphrase-TinyBERT-L6-v2',
        echecker_model='echecker_clotho_audiocaps_base',
        batch_size=32
    )
    
    # Convert to list of lists format
    list_refs = [[gt] for gt in results['ground_truth_captions']]
    
    # Batch evaluation (much faster)
    print("Encoding sentences and computing similarity...")
    fense_scores = evaluator.corpus_score(
        results['generated_captions'], 
        list_refs, 
        agg_score='none'
    )
    
    fense_scores = np.array(fense_scores)
    avg_fense = np.mean(fense_scores)
    
    print("\n" + "=" * 60)
    print("FENSE METRIC RESULTS")
    print("=" * 60)
    print(f"FENSE Score: {avg_fense:.4f}")
    print("=" * 60)
    
    results['metrics']['FENSE'] = float(avg_fense)
    results['metrics']['FENSE_individual'] = fense_scores.tolist()
    
except ImportError:
    print("FENSE not installed. Run: cd fense && pip install -e .")
except Exception as e:
    print(f"Error: {e}")

Computing FENSE metric...
Encoding sentences and computing similarity...
Encoding sentences


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Performing error detection


100%|██████████| 17/17 [00:00<00:00, 88.60it/s]


FENSE METRIC RESULTS
FENSE Score: 0.5828


In [ ]:
# Cell 13: Save Results (Optional)
import json
from datetime import datetime

# Save metrics to file
results_to_save = {
    'timestamp': datetime.now().isoformat(),
    'num_samples': len(results['generated_captions']),
    'metrics': {
        'BLEU_4': float(results['metrics']['BLEU_4']),
        'METEOR': float(results['metrics']['METEOR']),
        'ROUGE_L': float(results['metrics']['ROUGE_L']),
        'CIDEr': float(results['metrics']['CIDEr']),
        'SPIDER': float(results['metrics']['SPIDER']),
    },
    'config': {
        'use_beam_search': True,
        'max_length': 80,
        'min_length': 20,
    }
}

# Save to JSON
output_file = Path("evaluation_results.json")
with open(output_file, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"\nResults saved to {output_file}")
print(f"SPIDER Score: {results['metrics']['SPIDER']:.4f}")
